# 参数管理

In [3]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
net

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

## 参数访问

### 一次性访问所有参数


In [2]:
# 所有层的(没有ReLU)
print(net.state_dict())
print(type(net.state_dict()))
print('-----------')


# 某一层
print(net[2].state_dict())
print(type(net[2].state_dict()))

OrderedDict([('0.weight', tensor([[-0.1817,  0.2485, -0.4403,  0.0468],
        [-0.1051,  0.3449, -0.4999, -0.4464],
        [-0.3973,  0.1998,  0.2685,  0.3628],
        [-0.3525,  0.3066,  0.1421,  0.0981],
        [-0.2312, -0.3008,  0.1913, -0.0839],
        [ 0.0549,  0.0974, -0.3539,  0.0400],
        [ 0.2645, -0.0743,  0.2653,  0.4729],
        [ 0.3159,  0.1351, -0.3994,  0.0492]])), ('0.bias', tensor([ 0.4188, -0.4386,  0.3052,  0.0691,  0.2498, -0.1333,  0.4331, -0.2587])), ('2.weight', tensor([[ 0.0673, -0.0029,  0.1860,  0.1152, -0.1282, -0.0532,  0.0806,  0.1656]])), ('2.bias', tensor([-0.1891]))])
<class 'collections.OrderedDict'>
-----------
OrderedDict([('weight', tensor([[ 0.0673, -0.0029,  0.1860,  0.1152, -0.1282, -0.0532,  0.0806,  0.1656]])), ('bias', tensor([-0.1891]))])
<class 'collections.OrderedDict'>


类型为`<class 'collections.OrderedDict'>`, 这个全连接层包含两个参数，分别是该层的权重weight和偏置bias。两者都存储为单精度浮点数（float32）。

整体输出时名为`0.weight`,单层输出时名为`weight`

In [3]:
for name, param in net.named_parameters():
    print(name, param, sep='\n')
print('-----------')

for name, param in net[2].named_parameters():
    print(name, param, sep='\n')
print('-----------')

# 这个就是传入的不带名字的参数, torch.optim.SGD(net.parameters(), lr=0.1)
for param in net.parameters():
    print(param, sep='\n')

0.weight
Parameter containing:
tensor([[-0.1817,  0.2485, -0.4403,  0.0468],
        [-0.1051,  0.3449, -0.4999, -0.4464],
        [-0.3973,  0.1998,  0.2685,  0.3628],
        [-0.3525,  0.3066,  0.1421,  0.0981],
        [-0.2312, -0.3008,  0.1913, -0.0839],
        [ 0.0549,  0.0974, -0.3539,  0.0400],
        [ 0.2645, -0.0743,  0.2653,  0.4729],
        [ 0.3159,  0.1351, -0.3994,  0.0492]], requires_grad=True)
0.bias
Parameter containing:
tensor([ 0.4188, -0.4386,  0.3052,  0.0691,  0.2498, -0.1333,  0.4331, -0.2587],
       requires_grad=True)
2.weight
Parameter containing:
tensor([[ 0.0673, -0.0029,  0.1860,  0.1152, -0.1282, -0.0532,  0.0806,  0.1656]],
       requires_grad=True)
2.bias
Parameter containing:
tensor([-0.1891], requires_grad=True)
-----------
weight
Parameter containing:
tensor([[ 0.0673, -0.0029,  0.1860,  0.1152, -0.1282, -0.0532,  0.0806,  0.1656]],
       requires_grad=True)
bias
Parameter containing:
tensor([-0.1891], requires_grad=True)
-----------
Paramet

### 目标参数

每层的每个参数(`.weight`和`.bias`)都表示为参数类的一个实例 `<class 'torch.nn.parameter.Parameter'>`。


包含:

- 值: `.data`
      

- 梯度: `.grad`

  在上面这个网络中，由于我们还没有调用反向传播，所以参数的梯度处于初始状态`None`。

- 额外信息

  `.device`: 存储在CPU上还是GPU上

In [4]:
# 权重
print(type(net[2].weight))
print(net[2].weight)
# 值
print(net[2].weight.data)
# 梯度
print(net[2].weight.grad)
print()

# 偏置
print(type(net[2].bias))
print(net[2].bias)
# 值
print(net[2].bias.data)
# 梯度
print(net[2].bias.grad)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([[ 0.0673, -0.0029,  0.1860,  0.1152, -0.1282, -0.0532,  0.0806,  0.1656]],
       requires_grad=True)
tensor([[ 0.0673, -0.0029,  0.1860,  0.1152, -0.1282, -0.0532,  0.0806,  0.1656]])
None

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.1891], requires_grad=True)
tensor([-0.1891])
None


In [5]:
# 额外信息
print(net[2].weight.device)

cpu


In [6]:
# 另一种访问网络参数的方式
net.state_dict()['2.weight'].data

tensor([[ 0.0673, -0.0029,  0.1860,  0.1152, -0.1282, -0.0532,  0.0806,  0.1656]])

这为我们提供了另一种访问网络参数的方式，如下所示。


In [7]:
net.state_dict()['2.bias'].data

tensor([-0.1891])

### 从嵌套块收集参数

In [8]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1(), nn.Linear(4, 4), block1())
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))
net

Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (1): Linear(in_features=4, out_features=4, bias=True)
    (2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)

因为层是分层嵌套的，所以我们也可以像通过嵌套列表索引一样访问它们。
下面，我们访问第一个主要的块中、第二个子块的第一层的偏置项。


In [10]:
net[0][0][0].weight.data

tensor([[-0.1790, -0.0188, -0.1331,  0.1110],
        [ 0.2782, -0.3358,  0.3077,  0.4345],
        [-0.2253,  0.4150, -0.4980,  0.3908],
        [ 0.2091, -0.3623, -0.1977,  0.3252],
        [-0.0493,  0.0488, -0.2647,  0.3919],
        [-0.3323,  0.4749, -0.2896,  0.3591],
        [ 0.4975,  0.0167,  0.3238, -0.3183],
        [-0.1021, -0.3212, -0.0723, -0.0721]])

In [11]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1(), nn.Linear(4, 4), block1())
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))

# 检查参数是否相同
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])
net[0][0][0].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])

tensor([False, False, False, False])
tensor([False, False, False, False])


## 参数初始化

### 标准初始化

> 方法1

In [ ]:
def init_weights(m):
    if type(m) == nn.Linear:
        # 初始化 m.weight or m.bias
        # 以均值0和标准差0.01的正态分布
        nn.init.normal_(m.weight, mean=0, std=0.01)
        # 全0初始化
        nn.init.zeros_(m.bias)          
        # 均居分布
        nn.init.uniform_(m.weight, -10, 10)
        # 给定的常数
        nn.init.constant_(m.weight, 1)
        # xavier
        nn.init.xavier_uniform_(m.weight)


# 应用到网络上
net.apply(init_weights)

# 学习率为0.1的小批量随机梯度下降作为优化算法
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)

In [13]:
def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)


net.apply(init_weights)

Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (1): Linear(in_features=4, out_features=4, bias=True)
    (2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)

### 自定义初始化

> 方法1

在下面的例子中，我们使用以下的分布为任意权重参数$w$定义初始化方法：

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \text{ 可能性 } \frac{1}{4} \\
            0    & \text{ 可能性 } \frac{1}{2} \\
        U(-10, -5) & \text{ 可能性 } \frac{1}{4}
    \end{cases}
\end{aligned}
$$


同样，我们实现了一个`my_init`函数来应用到`net`。


In [15]:
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5


net.apply(init_weights)

Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (1): Linear(in_features=4, out_features=4, bias=True)
    (2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)

> 方法2: 直接设置参数。


In [16]:
net[1].weight.data[:] += 1
net[1].weight.data[0, 0] = 42
net[1].weight.data[0]

tensor([42.0000,  1.0000, -6.0076, -4.9941])

> 方法3: 

In [17]:
num_inputs, num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(
    num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(
    num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

optimizer = torch.optim.SGD(params, lr=0.01)

## 参数绑定

有时我们希望在多个层间共享参数：
我们可以定义一个稠密层，然后使用它的参数来设置另一个层的参数。


In [18]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


这个例子表明第三个和第五个神经网络层的参数是绑定的。
它们不仅值相等，而且由相同的张量表示。
因此，如果我们改变其中一个参数，另一个参数也会改变。
你可能会思考：当参数绑定时，梯度会发生什么情况？
答案是由于模型参数包含梯度，因此在反向传播期间第二个隐藏层
（即第三个神经网络层）和第三个隐藏层（即第五个神经网络层）的梯度会加在一起。


> 嵌套块的易錯区分

In [19]:
# 这不是共享的
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1(), nn.Linear(4, 4), block1())
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))

# 检查参数是否相同
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])
net[0][0][0].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])

tensor([False, False, False, False])
tensor([False, False, False, False])


In [20]:
# 这是共享的
block1 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                       nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1, nn.Linear(4, 4), block1)
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))

# 检查参数是否相同
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])
net[0][0][0].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[0][0][0].weight.data[0] == net[0][2][0].weight.data[0])

tensor([True, True, True, True])
tensor([True, True, True, True])


## 优化器, 学习率

> optimize总结

In [ ]:
torch.optim.SGD(net.parameters(), lr=lr)

# Adam优化器的主要吸引力在于它对初始学习率不那么敏感。
torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-3)

> 权重衰减


默认情况下，PyTorch同时衰减权重和偏移。

直接通过weight_decay指定weight decay超参数。这里我们只为权重设置了weight_decay，所以偏置参数不会衰减。

In [ ]:
# 偏置参数没有衰减
optimizer = torch.optim.SGD(
    [
        {"params": net[0].weight, 'weight_decay': wd},
        {"params": net[0].bias}
    ],
    lr=lr
)